# Day 3 · Notebook 2 — Student lab (client side)

Server-side exercises (adding endpoints, auth, logging) are in **`LAB.md`**.
These exercises only need a running API on `http://localhost:5000`.

Each exercise: try it first, then open the **✅ Solution** cell.

In [ ]:
import subprocess, sys, time, io
from pathlib import Path
import numpy as np, pandas as pd, requests
from PIL import Image

API = "http://127.0.0.1:5000"   # not "localhost": on Windows it resolves to IPv6 ::1 first -> +2 s per request
def api_up():
    try: return requests.get(f"{API}/health", timeout=1).ok
    except requests.ConnectionError: return False

server = None
if not api_up():
    log_file = open("api_server.log", "w")
    server = subprocess.Popen([sys.executable, "app.py"], cwd=Path("app/api").resolve(),
                              stdout=log_file, stderr=subprocess.STDOUT)
    for _ in range(60):
        if api_up() or server.poll() is not None: break
        time.sleep(0.5)
    if not api_up():
        log_file.close()
        raise RuntimeError("API failed to start:\n" + Path("api_server.log").read_text()
                           + f"\nKernel Python: {sys.executable} - is this the course venv?")
samples = sorted(Path("app/ui/samples").glob("*.png"))
def to_bytes(im):
    b = io.BytesIO(); im.save(b, format="PNG"); return b.getvalue()
print("API up:", api_up(), "|", len(samples), "samples")

## Exercise 1 (10 min) — Write a tiny Python client class

Write a class `CifarClient` with methods `health()`, `predict(path, model="int8")` and `compare(path)`.
It should raise a clear exception when the API returns a non-200 status.

In [ ]:
# TODO: your code here

### ✅ Solution (reveal after trying)

In [ ]:
class CifarClient:
    def __init__(self, base_url="http://localhost:5000", timeout=10):
        self.base, self.timeout = base_url.rstrip("/"), timeout
        self.s = requests.Session()             # re-uses TCP connections (keep-alive) - see Exercise 4

    def _check(self, r):
        if r.status_code != 200:
            raise RuntimeError(f"API error {r.status_code}: {r.json().get('error')}")
        return r.json()

    def health(self):
        return self._check(self.s.get(f"{self.base}/health", timeout=self.timeout))

    def predict(self, path, model="int8", top_k=3):
        with open(path, "rb") as f:
            return self._check(self.s.post(f"{self.base}/predict", params={"model": model, "top_k": top_k},
                                           files={"file": f}, timeout=self.timeout))

    def compare(self, path):
        with open(path, "rb") as f:
            return self._check(self.s.post(f"{self.base}/compare", files={"file": f}, timeout=self.timeout))

client = CifarClient()
print(client.health(), client.predict(samples[0])["prediction"])
try:
    client.predict(samples[0], model="nope")
except RuntimeError as e:
    print("caught:", e)

## Exercise 2 (15 min) — How robust is the model? Rotation & blur

For every sample image, apply (a) rotation by 0/15/30/45/90° and (b) Gaussian blur radius 0/1/2/3,
send to the API and compute accuracy per perturbation level for **both** fp32 and int8.
Does quantization make the model *less robust*?

Hint: `im.rotate(angle)`, `from PIL import ImageFilter; im.filter(ImageFilter.GaussianBlur(r))`

In [ ]:
# TODO: your code here

### ✅ Solution (reveal after trying)

In [ ]:
from PIL import ImageFilter
rows = []
for kind, levels, fn in [("rotate", [0, 15, 30, 45, 90], lambda im, a: im.rotate(a)),
                         ("blur", [0, 1, 2, 3], lambda im, r: im.filter(ImageFilter.GaussianBlur(r)))]:
    for lvl in levels:
        for model in ["fp32", "int8"]:
            correct = 0
            for p in samples:
                im = fn(Image.open(p).convert("RGB"), lvl)
                pred = requests.post(f"{API}/predict", params={"model": model},
                                     files={"file": to_bytes(im)}).json()["prediction"]
                correct += pred == p.stem.split("_")[0]
            rows.append({"perturbation": kind, "level": lvl, "model": model, "acc": correct / len(samples)})
rob = pd.DataFrame(rows)
rob.pivot_table(index=["perturbation", "level"], columns="model", values="acc")

## Exercise 3 (10 min) — Confidence threshold ("I'm not sure")

Write `safe_predict(path, threshold)` that returns `"unsure"` when the top-1 confidence is below `threshold`.
Using the sample images **plus** 10 random-noise images, pick a threshold that rejects most noise
while keeping most real images. Print a small table threshold → (% real accepted, % noise rejected).

In [ ]:
# TODO: your code here

### ✅ Solution (reveal after trying)

In [ ]:
rng = np.random.default_rng(0)
noise_imgs = [Image.fromarray((rng.random((32, 32, 3)) * 255).astype("uint8")) for _ in range(10)]
real_conf = [requests.post(f"{API}/predict", files={"file": p.read_bytes()}).json()["confidence"] for p in samples]
noise_conf = [requests.post(f"{API}/predict", files={"file": to_bytes(im)}).json()["confidence"] for im in noise_imgs]

table = []
for t in [0.3, 0.5, 0.6, 0.7, 0.8, 0.9]:
    table.append({"threshold": t,
                  "real_accepted_%": 100 * np.mean(np.array(real_conf) >= t),
                  "noise_rejected_%": 100 * np.mean(np.array(noise_conf) < t)})
print(pd.DataFrame(table))
print("\nNote: softmax confidence is a weak OOD signal - discuss why (Day 5 question bank).")

## Exercise 4 (10 min) — Measure, don't assume: connection re-use

"Re-using connections with `requests.Session()` is faster" — is it, **here**?
Measure the **median** latency of 50 requests for
(a) `GET /health` and (b) `POST /predict`, each (1) with a new `requests.get/post` per call (new TCP connection)
and (2) through one `requests.Session()` (HTTP keep-alive). Put the 4 numbers in a table. Explain what you see.

In [ ]:
# TODO: your code here

### ✅ Solution (reveal after trying)

In [ ]:
import statistics
payload = samples[0].read_bytes()
def median_ms(fn, n=50):
    fn()                                            # warm-up
    lat = []
    for _ in range(n):
        t0 = time.perf_counter(); fn(); lat.append((time.perf_counter() - t0) * 1000)
    return statistics.median(lat)

s = requests.Session()
res = pd.DataFrame({
    "new connection": {"GET /health": median_ms(lambda: requests.get(f"{API}/health")),
                       "POST /predict": median_ms(lambda: requests.post(f"{API}/predict", files={"file": payload}))},
    "keep-alive Session": {"GET /health": median_ms(lambda: s.get(f"{API}/health")),
                           "POST /predict": median_ms(lambda: s.post(f"{API}/predict", files={"file": payload}))},
}).round(1)
res

**What we measured on the reference Windows laptop** (numbers vary per machine — that's the point):

| Server | GET new → session | POST new → session |
|---|---|---|
| Flask **dev server** (`python app.py`, this notebook) | 6.4 → 5.8 ms | ~52 → ~61 ms (noisy) |
| **gunicorn** in Docker (Day 4 image) | 29.2 → 13.8 ms | 12.8 → 9.9 ms |

Take-aways:
* On `127.0.0.1` a TCP handshake costs almost nothing, so keep-alive barely matters against the dev server — and its
  POST numbers are noisy (small-packet effects such as Nagle's algorithm + delayed ACKs can add ~40 ms stalls).
  The dev server is not tuned for performance; that's one more reason not to use it in production.
* Against the production server (gunicorn behind Docker's port forwarding) the session **halves** GET latency.
* Over a real network the gap is much larger: every new HTTPS connection costs a TCP handshake **plus** a TLS
  handshake — 1–3 extra round-trips, i.e. tens to hundreds of ms per request. Clients that call an API repeatedly
  (like our Streamlit UI) should re-use a session.
* The general lesson: **measure on the real setup** — intuition about performance is often wrong.

In [ ]:
if server is not None:
    server.terminate(); server.wait(timeout=10); print("API stopped")